# LOSS
- 均方误差损失函数MSE
- 均方根误差损失函数RMSE
- 交叉熵损失函数Cross-Entropy
- 焦点损失函数Focal Loss

## 均方误差损失函数(mean squared error， MSE)

In [11]:
import torch
def MSE(y, y_hat):
    # y, y_hat:(batch)
    return ((y-y_hat) ** 2).mean()

y = torch.tensor([1.0, 0.0, 1.0])          # 真实标签: 样本1和3是正类，样本2是负类
y_hat = torch.tensor([0.9, 0.2, 0.8])      # 模型预测的概率
loss = MSE(y, y_hat)
print(loss)

# test
cr = torch.nn.MSELoss()
l = cr(y, y_hat)
print('torch:',l)

tensor(0.0300)
torch: tensor(0.0300)


## 均方根误差（rooted mean squared error，RMSE）
RMSE 与 MSE 非常接近，但是平方再开方的操作使得 RMSE 应当与具有相同的量纲，从直观上易于比较。我们可以简单认为，对于任意样本，模型预测的标签与真实值之间的偏差大致就等于 RMSE 的值。而 MSE 由于含有平方，其量纲和数量级相对来说不够直观，但其更容易求导。因此，我们常将 MSE 作为训练时的损失函数，而用 RMSE 作为模型的评价指标。

In [14]:
import torch
def RMSE(y, y_hat):
    # y, y_hat:(batch)
    return torch.sqrt((MSE(y, y_hat)))


y = torch.tensor([1.0, 0.0, 1.0])          # 真实标签: 样本1和3是正类，样本2是负类
y_hat = torch.tensor([0.9, 0.2, 0.8])      # 模型预测的概率
loss = RMSE(y, y_hat)
print(loss)


tensor(0.1732)


## 交叉熵损失函数（Cross-Entropy Loss）
在实际实现中，我们通常直接使用模型的原始 logits（未经过 sigmoid/softmax 的输出），并结合 log_softmax 来计算，以提高数值稳定性。

In [6]:
import torch
import torch.nn as nn
def BCE(y, y_hat, mode='mean'):
    # y, y_hat:(batch)
    # mean or sum
    if mode=='mean':
        return (- y * torch.log(y_hat) - (1-y) * torch.log(1-y_hat)).mean()
    if mode=='sum':
        return (- y * torch.log(y_hat) - (1-y) * torch.log(1-y_hat)).sum()

y = torch.tensor([1.0, 0.0, 1.0])          # 真实标签: 样本1和3是正类，样本2是负类
y_hat = torch.tensor([0.9, 0.2, 0.8])      # 模型预测的概率
loss = BCE(y, y_hat)
print(f"BCE Loss: {loss.item()}") # 输出: BCE Loss: 0.5545177459716797
# test
crit = nn.BCELoss()
l = crit(y, y_hat)
print(f"torch.BCE Loss: {loss.item()}")

BCE Loss: 0.18388253450393677
torch.BCE Loss: 0.18388253450393677


# 焦点损失函数(Focal Loss)Focal Loss 的统一形式为：

$$
\text{FL}(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)
$$

其中：
- $ p_t = \begin{cases} p, & y = 1 \\ 1 - p, & y = 0 \end{cases} $
- $ \alpha_t = \begin{cases} \alpha, & y = 1 \\ 1 - \alpha, & y = 0 \end{cases} $

- $ p \in [0, 1] $ 为模型预测的正类概率  
- $ y \in \{0, 1\} $ 为真实标签  
- $ \alpha \in [0, 1] $ 为平衡正负样本的权重参数（通常用于调节正样本的重要性）, $\alpha$越大，对正样本关注度越高，也算是一种对大量负样本的下采样
- $ \gamma \geq 0 $ 为聚焦参数（focusing parameter），控制难易样本的权重衰减程度，$\gamma$越大，对困难样本的关注度就越高

二分类交叉熵损失（BCE）的统一形式为：

$$
\text{BCE}(p_t) = -\log(p_t)
$$

其中：
- $ p_t = \begin{cases} p, & y = 1 \\ 1 - p, & y = 0 \end{cases} $

这里 $ p \in (0, 1) $ 是模型预测的正类概率，$ y \in \{0, 1\} $ 是真实标签。

In [ ]:
def focalloss(y, y_pred, alpha=0.25, gamma=2):
    p_t = y_pred * y + (1-y_pred) * (1-y)
    alpha_t = alpha * y + (1-alpha) * (1-y)
    return -alpha_t * (1-p_t) ** gamma * torch.log(p_t)

# KL散度

In [ ]:
import torch
def kl_divergence(p, q):
    return (p * torch.log(p / q)).sum()